In [2]:
import gymnasium as gym
import random

random.seed(1234) #seed pra gerar numeros

streets = gym.make("Taxi-v3", render_mode = "ansi").env #chama o taxi
streets.reset() 
print('\n' + streets.render()) #printa o taxi


+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [3]:
initial_state = streets.unwrapped.encode(2, 3, 2, 0) #altera onde o passageiro e o ponto de entrega esta

streets.unwrapped.s = initial_state

print('\n' + streets.render())


+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [4]:
streets.unwrapped.P[initial_state] #estado e oq ocorre se ir pra cada lado cada recompensa 

{0: [(1.0, 368, -1, False)],
 1: [(1.0, 168, -1, False)],
 2: [(1.0, 288, -1, False)],
 3: [(1.0, 248, -1, False)],
 4: [(1.0, 268, -10, False)],
 5: [(1.0, 268, -10, False)]}

In [5]:
import numpy as np

q_table = np.zeros([streets.observation_space.n, streets.action_space.n]) #tabela Q

learning_rate = 0.1 
discount_factor = 0.6
exploration = 0.1 
epochs = 10000 

for taxi_run in range(epochs): #roda 10000 vezes
    state = streets.reset()[0] #reseta
    done = False

    while not done: #enquanto nao acabar
        
        random_value = random.uniform(0, 1) #valor aleatoriio

        if (random_value < exploration): #se o valor aletorio for menor q o da exploracao
            action = streets.action_space.sample() #explora
        else:
            action = np.argmax(q_table[state]) #se nao ele vai pra tabela Q e pega a melhor acao

        next_state, reward, terminated, truncated, info = streets.step(action) #faz a acao

        done = terminated or truncated #acaba
        prev_q = q_table[state, action] #valor atual
        next_max_q = np.max(q_table[next_state]) #proximo melhor valor pro proximo estado 
        new_q = (1 - learning_rate) * prev_q + learning_rate * (reward + discount_factor * next_max_q) #novo Q, aprende e ve o futuro
        q_table[state, action] = new_q #atualiza a tabela

        state = next_state #vai pro proximo estado

In [6]:
q_table[initial_state]

array([-2.39512846, -2.40590892, -2.42513362, -2.3639511 , -7.21090727,
       -8.62772435])

In [9]:
from IPython.display import clear_output
from time import sleep

for tripnum in range(1, 11): #loop de 10 vezes
    state, _ = streets.reset() #reseta o mapa
    
    done = False
    
    while not done: #enquanto ainda nao terminou ele pega o melhor Q pra quele estado e manda fazer e dps termina
        action = np.argmax(q_table[state])

        next_state, reward, terminated, truncated, info = streets.step(action)
        done = terminated or truncated
        clear_output(wait = True)
        print("Trip number" + str(tripnum)) #nmr de vezes

        print(streets.render()) #mostra na tela o mapa
        
        sleep(.5) #delay
        state = next_state

    sleep(2)

Trip number10
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Dropoff)



# Your Challenge
Modify the block above to keep track of the total time steps, and use that as a metric as to how good our Q-learning system is. You might want to increase the number of simulated trips, and remove the sleep() calls to allow you to run over more samples

Now, try experimenting with the hyperparameters. How low can the number of epochs go before our model starts to suffer? Can you come up with better learning rates, discount factors, or exploration factors to make the training more efficient? The exploration vs exploitation rate in particular is interesting to experiment with.